# Tour des bibliothèques

Ce TP fait essayer un peu de chaque bibliothèque du chapitre : NumPy, SciPy, Matplotlib, pandas, scikit-learn et ElementTree. Les exercices sont indépendants, sauf quand ils reprennent les données d'un exercice précédent, et vont du plus simple au plus long : le dernier est facultatif. Toutes les données sont générées dans le notebook, il n'y a rien à télécharger.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

## 1. NumPy : indicer un tableau

Le tableau `durees` donne la durée de requêtes, en millisecondes : une ligne par serveur (4 serveurs), une colonne par heure (6 heures).

1. Affichez sa forme.
2. Affichez la durée du troisième serveur à la dernière heure, puis la ligne du premier serveur, puis la colonne de la deuxième heure.
3. Avec un masque booléen, affichez les durées supérieures à 200 ms, et comptez-les.
4. Calculez la durée moyenne de chaque serveur, puis trouvez le plus lent avec `argmax`.

**Documentation utile**

- [`numpy.ndarray.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.argmax.html)

In [ ]:
durees = np.array(
  [
    [120, 95, 310, 88, 140, 450],
    [110, 105, 130, 98, 125, 180],
    [240, 260, 230, 290, 310, 275],
    [90, 85, 100, 92, 88, 97],
  ]
)

# Votre code ici

### Conseils supplémentaires

Les indices commencent à 0 : le troisième serveur est la ligne 2, et `-1` désigne le dernier élément. `durees[:, 1]` prend toutes les lignes de la colonne 1. `axis=1` calcule le long des colonnes, donc une moyenne par ligne. Vous devez trouver 8 durées au-dessus de 200 ms, et le serveur d'indice 2 comme le plus lent, avec une moyenne de 267,5 ms.

```python
print(durees.shape)
print(durees[2, -1])
print(durees[0])
print(durees[:, 1])
lentes = durees[...]
print(lentes, len(lentes))
moyennes = durees.mean(axis=...)
print(moyennes, moyennes.argmax())
```

### Solution

In [ ]:
print("Forme :", durees.shape)
print("Troisième serveur, dernière heure :", durees[2, -1])
print("Premier serveur :", durees[0])
print("Deuxième heure :", durees[:, 1])
lentes = durees[durees > 200]
print("Durées supérieures à 200 ms :", lentes)
print("Nombre :", len(lentes))
moyennes = durees.mean(axis=1)
print("Moyennes :", moyennes)
print("Serveur le plus lent :", moyennes.argmax())

## 2. SciPy : tester et optimiser

Les listes `avant` et `apres` donnent des durées de requêtes, en millisecondes, avant et après un déploiement.

1. Résumez `avant` avec `stats.describe`, et affichez les deux moyennes.
2. Comparez les deux séries avec `stats.ttest_ind`, sans supposer leurs variances égales. Au seuil de 5 %, la baisse est-elle réelle ?
3. La fonction `cout` donne le coût d'exploitation selon le nombre de serveurs `n`. Avec `optimize.minimize_scalar`, trouvez le nombre de serveurs qui le minimise, entre 1 et 50.
4. Un nombre de serveurs est entier : comparez `cout` aux deux entiers les plus proches.

**Documentation utile**

- [`scipy.stats.describe`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.describe.html)

In [ ]:
avant = [132, 118, 125, 140, 151, 122, 137, 129, 145, 120]
apres = [118, 110, 127, 105, 121, 115, 108, 124, 112, 119]


def cout(n: float) -> float:
  return 80 * n + 5000 / n


# Votre code ici

### Conseils supplémentaires

`equal_var=False` demande le test de Welch, qui ne suppose pas les variances égales. Le résultat a un champ `pvalue`. `bounds=(1, 50)` borne la recherche, et le résultat donne la position du minimum dans `.x`, sa valeur dans `.fun`. Vous devez trouver une p-value d'environ 0,0016, et un optimum à 7,91 serveurs pour un coût de 1264,9.

```python
print(stats.describe(avant))
print(np.mean(avant), np.mean(apres))
test = stats.ttest_ind(avant, apres, equal_var=...)
print(test.pvalue)
optimum = optimize.minimize_scalar(cout, bounds=...)
print(optimum.x, optimum.fun)
print(cout(7), cout(8))
```

### Solution

In [ ]:
print(stats.describe(avant))
print("Moyenne avant :", np.mean(avant))
print("Moyenne après :", np.mean(apres))
test = stats.ttest_ind(avant, apres, equal_var=False)
print(f"p-value : {test.pvalue:.4f}")
optimum = optimize.minimize_scalar(cout, bounds=(1, 50))
print(f"Optimum : {optimum.x:.2f} serveurs, coût {optimum.fun:.1f}")
print(f"7 serveurs : {cout(7):.1f}, 8 serveurs : {cout(8):.1f}")

La p-value est sous 0,05 : un tel écart serait peu probable si les deux moyennes étaient égales, la baisse est donc réelle à ce seuil. Parmi les entiers, 8 serveurs coûtent le moins.

## 3. Matplotlib : deux graphiques

Avec le tableau `durees` de l'exercice 1 :

1. Créez une figure de deux graphiques côte à côte, avec `plt.subplots`.
2. À gauche, tracez une courbe par serveur, avec une légende, et une ligne horizontale au seuil de 200 ms. Nommez les axes.
3. À droite, tracez l'histogramme de toutes les durées, avec 10 classes. Nommez les axes.
4. Donnez un titre à chaque graphique, et enregistrez la figure dans `durees.png`.

**Documentation utile**

- [`Axes.hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.hist.html)
- [`Axes.set_title`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_title.html)
- [`numpy.ndarray.ravel`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.ravel.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`plt.subplots(1, 2, figsize=(10, 4))` renvoie la figure et un tableau de deux `Axes`. `enumerate(durees)` parcourt les lignes avec leur indice, pour la légende. `durees.ravel()` met le tableau à plat pour l'histogramme, et `bins=10` fixe le nombre de classes.

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ligne in enumerate(durees):
  axes[0].plot(ligne, marker="o", label=f"serveur {i}")
axes[0].axhline(...)
axes[0].legend()
axes[1].hist(durees.ravel(), bins=...)
fig.savefig("durees.png")
```

### Solution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ligne in enumerate(durees):
  axes[0].plot(ligne, marker="o", label=f"serveur {i}")
axes[0].axhline(200, color="red", linestyle="--", label="seuil")
axes[0].set_xlabel("heure")
axes[0].set_ylabel("durée (ms)")
axes[0].set_title("Durées (ms) par serveur")
axes[0].legend()
axes[1].hist(durees.ravel(), bins=10)
axes[1].set_xlabel("durée (ms)")
axes[1].set_ylabel("nombre de requêtes")
axes[1].set_title("Répartition des durées")
fig.savefig("durees.png")

## 4. pandas : filtrer et grouper

Le `DataFrame` `requetes` décrit 300 requêtes simulées : leur service, la charge du serveur, leur durée et leur statut HTTP.

1. Regardez les premières lignes, et résumez les colonnes numériques avec `describe`.
2. Comptez les requêtes de chaque statut avec `value_counts`.
3. Gardez les requêtes en erreur (statut 500) de plus de 300 ms, et comptez-les.
4. Ajoutez une colonne `secondes`, la durée en secondes.
5. Calculez la durée moyenne de chaque service.

**Documentation utile**

- [`DataFrame.head`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)
- [`DataFrame.describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

In [ ]:
rng = np.random.default_rng(0)
charge = rng.integers(1, 60, size=300)
requetes = pd.DataFrame(
  {
    "service": rng.choice(["api", "web", "batch"], size=300),
    "charge": charge,
    "duree_ms": (60 + 6.5 * charge + rng.normal(0, 20, size=300)).round(),
    "statut": rng.choice([200, 404, 500], size=300, p=[0.85, 0.1, 0.05]),
  }
)

# Votre code ici

### Conseils supplémentaires

Dans une cellule, seule la dernière expression s'affiche : utilisez `print` ou `display` pour les autres. Une condition sur deux colonnes s'écrit avec `&`, chaque comparaison entre parenthèses. `requetes.groupby("service")["duree_ms"].mean()` donne une moyenne par service. Vous devez trouver 5 requêtes en erreur de plus de 300 ms.

```python
print(requetes.head())
print(requetes.describe())
print(requetes["statut"].value_counts())
lentes = requetes[(...) & (...)]
print(len(lentes))
requetes["secondes"] = ...
print(requetes.groupby("service")["duree_ms"].mean())
```

### Solution

In [ ]:
print(requetes.head())
print(requetes.describe())
print(requetes["statut"].value_counts())
lentes = requetes[(requetes["statut"] == 500) & (requetes["duree_ms"] > 300)]
print("Requêtes lentes en erreur :", len(lentes))
requetes["secondes"] = requetes["duree_ms"] / 1000
print(requetes.groupby("service")["duree_ms"].mean())

## 5. scikit-learn : prédire une durée

Avec le `DataFrame` `requetes` de l'exercice précédent, prédisez la durée d'une requête d'après la charge. `X` et `y` sont donnés.

1. Séparez les données en apprentissage et test avec `train_test_split`, avec `random_state=0`.
2. Entraînez une `LinearRegression`, et affichez son score sur le test.
3. Lisez sa pente et son ordonnée à l'origine, à comparer à la formule de la simulation, puis prédisez la durée pour une charge de 30.
4. Remplacez le modèle par un `DecisionTreeRegressor`, avec `random_state=0`, et comparez ses scores sur l'apprentissage et sur le test.

**Documentation utile**

- [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [`DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)

In [ ]:
X = requetes[["charge"]].to_numpy()
y = requetes["duree_ms"].to_numpy()

# Votre code ici

### Conseils supplémentaires

`train_test_split(X, y, random_state=0)` renvoie quatre tableaux, dans l'ordre `X_train, X_test, y_train, y_test`. Le modèle entraîné a les attributs `coef_` (un tableau d'une pente par colonne de `X`) et `intercept_`. `predict` attend un tableau 2D : `[[30]]`, une ligne, une colonne. La simulation utilise `60 + 6.5 * charge`, et la prédiction pour une charge de 30 doit donc être proche de 255 ms.

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
modele = LinearRegression().fit(...)
print(modele.score(X_test, y_test))
print(modele.coef_[0], modele.intercept_)
print(modele.predict([[30]])[0])
arbre = DecisionTreeRegressor(random_state=0).fit(...)
print(arbre.score(X_train, y_train), arbre.score(X_test, y_test))
```

### Solution

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
modele = LinearRegression().fit(X_train, y_train)
print(f"R² test : {modele.score(X_test, y_test):.3f}")
print(f"Pente : {modele.coef_[0]:.2f}, ordonnée à l'origine : {modele.intercept_:.1f}")
print(f"Charge 30 : {modele.predict([[30]])[0]:.0f} ms")
arbre = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print(
  f"Arbre, apprentissage : {arbre.score(X_train, y_train):.3f}, "
  f"test : {arbre.score(X_test, y_test):.3f}"
)

Changer d'algorithme n'a demandé que de changer de classe. L'arbre obtient un meilleur score sur l'apprentissage que sur le test, où il fait moins bien que la droite : seul le score sur le test estime ce qu'un modèle vaudra sur de nouvelles requêtes.

## 6. ElementTree : lire du XML

La chaîne `journal` contient un journal de requêtes au format XML.

1. Analysez-la avec `ET.fromstring`, et affichez le nom de la racine et son nombre d'enfants.
2. Parcourez les requêtes, et affichez pour chacune son service et sa durée.
3. Avec `findall`, gardez les requêtes en erreur (statut 500), et calculez leur durée moyenne.
4. Comptez les requêtes de chaque service avec `Counter`.

**Documentation utile**

- [`collections.Counter`](https://docs.python.org/fr/3/library/collections.html#collections.Counter)

In [ ]:
journal = """<journal>
  <requete service="api" statut="200"><duree>120</duree></requete>
  <requete service="web" statut="200"><duree>95</duree></requete>
  <requete service="web" statut="500"><duree>310</duree></requete>
  <requete service="api" statut="200"><duree>88</duree></requete>
  <requete service="batch" statut="200"><duree>640</duree></requete>
  <requete service="api" statut="500"><duree>450</duree></requete>
</journal>"""

# Votre code ici

### Conseils supplémentaires

`racine.tag` donne le nom, `len(racine)` le nombre d'enfants. Un attribut se lit avec `.get("service")`, le texte d'un enfant avec `.findtext("duree")`, toujours sous forme de chaîne : convertissez-le avec `int`. Le chemin `"requete[@statut='500']"` filtre sur un attribut. Vous devez trouver une durée moyenne de 380 ms pour les erreurs, et 3 requêtes pour `api`.

```python
racine = ET.fromstring(journal)
print(racine.tag, len(racine))
for requete in racine:
  print(requete.get(...), requete.findtext(...))
erreurs = racine.findall(...)
print(np.mean([int(e.findtext("duree")) for e in erreurs]))
print(Counter(requete.get("service") for requete in racine))
```

### Solution

In [ ]:
racine = ET.fromstring(journal)
print("Racine :", racine.tag)
print("Requêtes :", len(racine))
for requete in racine:
  print(requete.get("service"), int(requete.findtext("duree")))
erreurs = racine.findall("requete[@statut='500']")
print(
  "Durée moyenne des erreurs :", np.mean([int(e.findtext("duree")) for e in erreurs])
)
print(Counter(requete.get("service") for requete in racine))

## 7. Pour finir : du journal XML au graphique

Cet exercice facultatif enchaîne trois bibliothèques. La cellule ci-dessous écrit `journal.xml`, le journal d'une journée : 500 requêtes, avec leur service, leur heure, leur statut et leur durée.

1. Lisez le fichier avec `ET.parse`, et construisez un `DataFrame` d'une ligne par requête, avec des colonnes numériques pour l'heure, le statut et la durée.
2. Gardez les requêtes réussies (statut 200), et calculez leur durée moyenne pour chaque heure.
3. Tracez cette moyenne en fonction de l'heure, avec Matplotlib. À quelle heure le service est-il le plus lent ?

**Documentation utile**

- [`ET.parse`](https://docs.python.org/fr/3/library/xml.etree.elementtree.html#xml.etree.ElementTree.parse)
- [`ElementTree.getroot`](https://docs.python.org/fr/3/library/xml.etree.elementtree.html#xml.etree.ElementTree.ElementTree.getroot)

In [ ]:
rng = np.random.default_rng(1)
racine = ET.Element("journal")
for _ in range(500):
  heure = int(rng.integers(0, 24))
  charge = 10 + 40 * np.exp(-((heure - 14) ** 2) / 18)
  requete = ET.SubElement(
    racine,
    "requete",
    service=str(rng.choice(["api", "web"])),
    heure=str(heure),
    statut=str(rng.choice([200, 500], p=[0.9, 0.1])),
  )
  ET.SubElement(requete, "duree").text = str(
    round(60 + 6.5 * charge + rng.normal(0, 20))
  )
ET.ElementTree(racine).write("journal.xml", encoding="utf-8", xml_declaration=True)

# Votre code ici

### Conseils supplémentaires

`ET.parse("journal.xml").getroot()` donne la racine. Une liste de dictionnaires, un par requête, se passe directement à `pd.DataFrame`. Convertissez les nombres avec `int` en lisant le XML. Ensuite, `groupby("heure")["duree_ms"].mean()` donne une `Series` dont l'index est l'heure, à passer à `ax.plot(par_heure.index, par_heure)`. Le maximum est vers 14 h.

```python
racine = ET.parse("journal.xml").getroot()
lignes = [
  {
    "heure": int(requete.get("heure")),
    "statut": ...,
    "duree_ms": ...,
  }
  for requete in racine
]
journal_df = pd.DataFrame(lignes)
ok = journal_df[...]
par_heure = ok.groupby("heure")["duree_ms"].mean()
fig, ax = plt.subplots()
ax.plot(par_heure.index, par_heure, marker="o")
```

### Solution

In [ ]:
racine = ET.parse("journal.xml").getroot()
lignes = [
  {
    "service": requete.get("service"),
    "heure": int(requete.get("heure")),
    "statut": int(requete.get("statut")),
    "duree_ms": int(requete.findtext("duree")),
  }
  for requete in racine
]
journal_df = pd.DataFrame(lignes)
ok = journal_df[journal_df["statut"] == 200]
par_heure = ok.groupby("heure")["duree_ms"].mean()
fig, ax = plt.subplots()
ax.plot(par_heure.index, par_heure, marker="o")
ax.set_xlabel("heure")
ax.set_ylabel("durée (ms)")
print(par_heure.idxmax())